# UMAP

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load training and test datasets

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico256.ens.nc'
ds    = xr.open_dataset(fname)
train = ds['tephra_col_mass']

fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Load a pre-trained VAE

In [ ]:
import torch
from torch.utils.data import DataLoader
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale, EnsembleDataset

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Define datasets and dataloaders

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

## Datasets
train_ds = EnsembleDataset(train, transform)
test_ds  = EnsembleDataset(test, transform)

## Dataloaders
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)

## Encoding into the latent space

In [ ]:
z_train = []
z_test = []

model.eval()
with torch.no_grad():
    for batch in train_loader:
        mu, logvar = model.encode(batch)
        z = model.reparameterize(mu, logvar)
        z_train.append(z)

    for batch in test_loader:
        mu, logvar = model.encode(batch)
        z = model.reparameterize(mu, logvar)
        z_test.append(z)

z_train = torch.cat(z_train, dim=0)
z_test  = torch.cat(z_test, dim=0)

## Covariance of the encoded training dataset

In [ ]:
import pandas as pd

df_train = pd.DataFrame(z_train)
zcov = df_train.cov()

## UMAP projection

In [ ]:
import numpy as np

# Create a full array concatenating: z_train and z_test
z_full = np.concatenate(
    [
        z_train.numpy(),
        z_test.numpy()
    ], axis=0
)

In [ ]:
### Requires: pip install umap-learn
import umap

# Fit UMAP on all data:
# fitting on train+test together keeps both sets in the same 2D coordinate system,
# so their embeddings can be directly compared
reducer = umap.UMAP(
    n_neighbors  = 15,             # how many nearby points UMAP considers per point;
    min_dist     = 0.2,            # minimum distance between points in the 2D embedding;
    n_components = 2,              # output dimensionality (2D, for plotting)
    metric       = "euclidean",    # distance metric used in the original latent space
    random_state = 42,             # fixes randomness so the embedding is reproducible
    #n_jobs=-1                     # Use all cores. Require disabling random_state
)

In [ ]:
# latent_all should be latent_train and latent_test stacked together
z_full_2d = reducer.fit_transform(z_full)

# Split back into train and test embeddings
n_train = len(z_train)
z_train_2d = z_full_2d[:n_train]  # first n_train rows = train points
z_test_2d  = z_full_2d[n_train:]  # remaining rows = test points

## Load ensemble perturbation

In [ ]:
df_train = pd.read_csv('data/perturbation256.csv',  sep=r'\s+', index_col=False)
df_test  = pd.read_csv('data/perturbation2048.csv', sep=r'\s+', index_col=False)

df_train['UMAP-1'] = z_train_2d[:,0]
df_train['UMAP-2'] = z_train_2d[:,1]

df_test['UMAP-1'] = z_test_2d[:,0]
df_test['UMAP-2'] = z_test_2d[:,1]

## Plot UMAP projection and covariance

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=2, figsize=(13,17))

fig.subplots_adjust(hspace=0.05)

####### Covariance matrix ########
cax = axs[0].matshow(zcov, vmin=-1, vmax=1, cmap = 'managua')
axs[0].set_aspect('equal')
axs[0].set_title('(a) Covariance between latent vector components')
fig.colorbar(cax, ax=axs[0], 
             orientation = 'horizontal', 
             shrink= 0.5,
             pad=0.02)

####### UMAP ########
cs = axs[1].scatter(x='UMAP-1', y='UMAP-2', 
                    c     = 'COLUMN_HEIGHT', 
                    cmap  = 'coolwarm', 
                    alpha = 0.5,
                    data  = df_test, 
                    label = 'Test (2048 samples)',
                   )
axs[1].scatter(x='UMAP-1', y='UMAP-2', 
               marker = '+', 
               c      = 'k', 
               data   = df_train, 
               label  = 'Training (256 samples)'
              )

axs[1].set_aspect('equal')
axs[1].set(title='(b) UMAP Projection of Latent Space',
           xlabel='UMAP-1',
           ylabel='UMAP-2')
cbar = fig.colorbar(cs, 
                    label       = 'Column height',
                    orientation = 'horizontal', 
                    shrink      = 0.5,
                    pad         = 0.1,
                    ax          = axs[1], 
                   )
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
cbar.set_ticklabels(['Very low','low','middle','high','Very high'])
axs[1].legend(ncol=2, shadow=True)